In [ ]:
#script para verificar a vriancia do dataset por origem 
import pandas as pd
import os
import numpy as np

In [ ]:
# Here I combine datasets with the same source (e.g., ce-*)
def combine_datasets(path, substring):
    files = [f for f in os.listdir(path) if substring in f and f.endswith('.csv')]
    
    if not files:
        print(f"No files containing the substring '{substring}' were found.")
        return None
    
    dataframe_list = []
    for file in files:
        file_path = os.path.join(path, file)
        df = pd.read_csv(file_path)
        dataframe_list.append(df)
    
    combined_df = pd.concat(dataframe_list, ignore_index=True)
    return combined_df

path = '../../datasets/serie-multivariada'
dataframes_by_source = {}

for dirs, root, files in os.walk(path):
    for file in files:
        if file.endswith('.csv'):
            source = '-' + file.split('-')[2] + '-'
            #print(f"Processing files with source: {source}")
            df = combine_datasets(path, source)
            
            if df is not None:
                key_name = f"{source.strip('-')}" 
                dataframes_by_source[key_name] = df


In [ ]:
#carrego meu dataset com os rmse de cada modelo por link de origem 
rmse_bbr =pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
rmse_cubic =pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_cubic_rmse.csv')

In [ ]:
# variancia 

protocol = 'Vazao_bbr'
rmse = rmse_bbr

correlations_by_model = {}

variances = {}
for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
        
    variance = dataset[[protocol]].var().iloc[0]
    variances[key] = variance

models = rmse.columns[1:]  

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance': rmse['source'].map(variances)
    })
    correlation = df_model[['RMSE', 'Variance']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model} - BBR: {correlation:.4f}")


In [ ]:
# Coeficiente de variância dos valores de vazao 

protocol = 'Vazao_bbr'
rmse = rmse_bbr

correlations_by_model = {}
variances = {}

for key, value in dataframes_by_source.items():
    dataset = dataframes_by_source[key]
    variance_coef = (dataset[protocol].std() / dataset[protocol].mean()) * 100
    variances[key] = variance_coef

models = rmse.columns[1:]

for model in models:
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])  
    })
    correlation = df_model[['RMSE', 'Variance_coef']].corr().iloc[0, 1]
    correlations_by_model[model] = correlation

# Exibindo as correlações
print("Correlation between high RMSE and dataset variance for each model:")
for model, correlation in correlations_by_model.items():
    print(f"{model}: {correlation:.4f}")



In [ ]:
#gerando arquivos csv com coeficiente de variancia e nrmse de todos os modelos e links 

# def process_rmse_and_variances(rmse_file, dataframes_by_source, target):
#     # Certifique-se de que rmse_file seja um DataFrame
#     if isinstance(rmse_file, str):  # Se for string, carregue como CSV
#         rmse_file = pd.read_csv(rmse_file)
#     variances = {}
#     for key, _ in dataframes_by_source.items():
#         dataset = dataframes_by_source[key]
#         variance_coef = (dataset[target].std() / dataset[target].mean()) * 100
#         variances[key] = variance_coef

#     models = rmse_file.columns[1:]
#     final_df = pd.DataFrame({'Source': rmse_file['source']})
#     final_df['Variance_coef'] = final_df['Source'].map(lambda x: variances.get(x, None))
#     for model in models:
#         final_df[f'RMSE_{model}'] = rmse_file[model]
#     output_path = f'../../results/regression/coefVariation_nrmse/Coefvariation_nrmse_{target}.csv'
#     final_df.to_csv(output_path, index=False)
#     return final_df

# rmse = pd.read_csv('../../results/regression/predictions-regression-bysource/regression_Vazao_bbr_rmse.csv')
# result = process_rmse_and_variances(rmse, dataframes_by_source, 'Vazao_bbr')



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Convertendo o dicionário em DataFrame
correlations_df = pd.DataFrame(list(correlations_by_model.items()), 
                             columns=['Model', 'Correlation'])
correlations_df = correlations_df.sort_values('Correlation', ascending=True)

# Configurando o tema do seaborn
sns.set_theme(style="whitegrid")

# Criando os gráficos
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 1. Gráfico de barras horizontal usando barplot do seaborn
sns.barplot(y='Model', x='Correlation', data=correlations_df, ax=ax1,
           palette='viridis', orient='h')
ax1.set_title('Correlation between RMSE and Dataset Variance by Model')
ax1.set_xlabel('Correlation Coefficient')

# 2. Heatmap das correlações
heatmap_data = correlations_df.set_index('Model')
sns.heatmap(heatmap_data.T, annot=True, cmap='RdYlBu', center=0, 
            fmt='.3f', ax=ax2)
ax2.set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

# Scatter plots para cada modelo
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for idx, model in enumerate(correlations_by_model.keys()):
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])
    })
    
    sns.scatterplot(data=df_model, x='Variance_coef', y='RMSE', ax=axes[idx])
    axes[idx].set_title(f'{model}\nCorr: {correlations_by_model[model]:.3f}')
    axes[idx].set_xlabel('Variance Coefficient (%)')
    axes[idx].set_ylabel('RMSE')

plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(12, 6))
rmse_melted = rmse.melt(id_vars=['source'], 
                        var_name='Model', 
                        value_name='RMSE')
sns.boxplot(x='Model', y='RMSE', data=rmse_melted, palette='viridis')
plt.xticks(rotation=45)
plt.title('Distribution of RMSE by Model')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Configuração para exibir mais casas decimais no pandas
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# 1. Tabela de correlações
print("\nCorrelações entre RMSE e Variância do Dataset por Modelo:")
print(correlations_df.to_string(index=False))

# 2. Tabela com estatísticas descritivas do RMSE por modelo
rmse_stats = rmse_melted.groupby('Model')['RMSE'].agg([
    ('Média', 'mean'),
    ('Mediana', 'median'),
    ('Desvio Padrão', 'std'),
    ('Mínimo', 'min'),
    ('Máximo', 'max')
]).round(4)

print("\nEstatísticas Descritivas do RMSE por Modelo:")
print(rmse_stats.to_string())

# 3. Tabela cruzada de RMSE médio por modelo
pivot_table = pd.pivot_table(
    rmse_melted,
    values='RMSE',
    index='Model',
    aggfunc=['mean', 'std']
).round(4)

print("\nRMSE Médio e Desvio Padrão por Modelo:")
print(pivot_table.to_string())

# 4. Tabela de variâncias por fonte
variances_df = pd.DataFrame.from_dict(variances, orient='index', 
                                    columns=['Coeficiente de Variação'])
variances_df = variances_df.round(4)

print("\nCoeficiente de Variação por Fonte de Dados:")
print(variances_df.to_string())

# Visualizações
# Gráfico de correlações e heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

sns.barplot(y='Model', x='Correlation', data=correlations_df, ax=ax1,
           palette='viridis', orient='h')
ax1.set_title('Correlation between RMSE and Dataset Variance by Model')
ax1.set_xlabel('Correlation Coefficient')

heatmap_data = correlations_df.set_index('Model')
sns.heatmap(heatmap_data.T, annot=True, cmap='RdYlBu', center=0, 
            fmt='.3f', ax=ax2)
ax2.set_title('Correlation Heatmap')

plt.tight_layout()
plt.show()

# Scatter plots
fig, axes = plt.subplots(3, 4, figsize=(20, 15))
axes = axes.ravel()

for idx, model in enumerate(correlations_by_model.keys()):
    df_model = pd.DataFrame({
        'Source': rmse['source'],
        'RMSE': rmse[model],
        'Variance_coef': rmse['source'].map(lambda x: variances[x])
    })
    
    sns.scatterplot(data=df_model, x='Variance_coef', y='RMSE', ax=axes[idx])
    axes[idx].set_title(f'{model}\nCorr: {correlations_by_model[model]:.3f}')
    axes[idx].set_xlabel('Variance Coefficient (%)')
    axes[idx].set_ylabel('RMSE')

plt.tight_layout()
plt.show()

# Boxplot
plt.figure(figsize=(12, 6))
sns.boxplot(x='Model', y='RMSE', data=rmse_melted, palette='viridis')
plt.xticks(rotation=45)
plt.title('Distribution of RMSE by Model')
plt.tight_layout()
plt.show()